# 01. Loading Models and Tokenizers

**Topics covered:** AutoModel · AutoTokenizer · Model Cards · Device Placement

This notebook starts the **Hugging Face basics** series. After building a GPT from scratch, we now learn how to load production-ready models and tokenizers from the Hugging Face Hub.

We will:
1. Understand **Model Cards** and how to read them
2. Load tokenizers with **AutoTokenizer**
3. Load models with **AutoModel** (and task-specific variants)
4. Move models to the correct **device** (CPU / CUDA)
5. Inspect model config, parameters, and run a simple forward pass

## 1. Setup & Imports

```bash
pip install transformers torch # run once if needed
```

In [23]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    AutoConfig,
    pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
except ImportError:
    print("Please install: pip install transformers")

Using device: cpu
PyTorch version: 2.11.0+cpu
Transformers version: 5.16.1


## 2. Model Cards

Every model on the [Hugging Face Hub](https://huggingface.co/models) has a **Model Card** — a README that describes:

- Intended use and limitations
- Training data and procedure
- Evaluation results
- Ethical considerations
- How to load the model (code snippets)

### Useful fields you will see

| Field | Meaning |
|-------|--------|
| `model_type` | Architecture family (gpt2, llama, bert, …) |
| `architectures` | Concrete class name(s) |
| `vocab_size` | Size of the tokenizer vocabulary |
| `hidden_size` / `n_embd` | Model dimension |
| `num_attention_heads` | Number of attention heads |
| `num_hidden_layers` | Depth |
| `max_position_embeddings` | Context length |

Always read the model card before using a model in production.

In [24]:
# Load only the configuration (very fast, no weights)
model_name = "gpt2"  # small, widely available #bert-base-uncased

config = AutoConfig.from_pretrained(model_name)
print("Model type      :", config.model_type)
print("Architectures   :", config.architectures)
print("Vocab size      :", config.vocab_size)
print("Hidden size     :", config.n_embd)
print("Num layers      :", config.n_layer)
print("Num heads       :", config.n_head)
print("Max positions   :", config.n_positions)
print("\nFull config:\n", config)

Model type      : gpt2
Architectures   : ['GPT2LMHeadModel']
Vocab size      : 50257
Hidden size     : 768
Num layers      : 12
Num heads       : 12
Max positions   : 1024

Full config:
 GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "ma

## 3. AutoTokenizer

`AutoTokenizer` automatically selects the correct tokenizer class for a given model checkpoint.

It handles:
- Token → ID conversion and back
- Special tokens (`[CLS]`, `[SEP]`, `<|endoftext|>`, …)
- Padding, truncation, attention masks
- Chat templates (for instruction-tuned models)

In [25]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer class :", tokenizer.__class__.__name__)
print("Vocab size      :", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)
print("Pad token       :", tokenizer.pad_token)
print("EOS token       :", tokenizer.eos_token)
print("BOS token       :", tokenizer.bos_token)

# GPT-2 has no pad token by default – a common fix:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("\nSet pad_token = eos_token")

Tokenizer class : GPT2Tokenizer
Vocab size      : 50257
Model max length: 1024
Pad token       : None
EOS token       : <|endoftext|>
BOS token       : <|endoftext|>

Set pad_token = eos_token


In [26]:
text = "Hello, how are you doing today?"

# Encode
encoded = tokenizer(text)
print("Encoded (dict):", encoded)

ids = tokenizer.encode(text)
print("Token IDs     :", ids)
print("Tokens        :", tokenizer.convert_ids_to_tokens(ids))

# Decode
print("Decoded       :", tokenizer.decode(ids))

# Batch encoding with padding & truncation
batch = tokenizer(
    ["Short text.", "A much longer piece of text that will be truncated."],
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt",
)
print("\nBatch input_ids shape :", batch["input_ids"].shape)
print("Batch attention_mask  :", batch["attention_mask"])

Encoded (dict): {'input_ids': [15496, 11, 703, 389, 345, 1804, 1909, 30], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Token IDs     : [15496, 11, 703, 389, 345, 1804, 1909, 30]
Tokens        : ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', 'Ġdoing', 'Ġtoday', '?']
Decoded       : Hello, how are you doing today?

Batch input_ids shape : torch.Size([2, 12])
Batch attention_mask  : tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## 4. AutoModel and Task-Specific Classes

| Class | Purpose |
|-------|--------|
| `AutoModel` | Base model (hidden states only) |
| `AutoModelForCausalLM` | Language modeling / generation (GPT-style) |
| `AutoModelForMaskedLM` | Masked language modeling (BERT-style) |
| `AutoModelForSequenceClassification` | Classification heads |
| `AutoModelForTokenClassification` | NER, POS tagging, … |
| `AutoModelForQuestionAnswering` | Extractive QA |

`Auto*` classes read the model card / config and instantiate the right architecture.  *(Footer/end of notebook)

In [27]:
# Base model – returns last_hidden_state
base_model = AutoModel.from_pretrained(model_name)
print("Base model class:", base_model.__class__.__name__)
print("Base model config.hidden_size:", base_model.config.n_embd)

# Causal LM – returns logits over vocabulary
lm_model = AutoModelForCausalLM.from_pretrained(model_name)
print("\nCausal LM class:", lm_model.__class__.__name__)
print("LM head output features:", lm_model.lm_head.out_features)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Base model class: GPT2Model
Base model config.hidden_size: 768


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Causal LM class: GPT2LMHeadModel
LM head output features: 50257


In [28]:
# Forward pass with the base model
inputs = tokenizer("The future of AI is", return_tensors="pt")
print("Input IDs:", inputs["input_ids"])

with torch.no_grad():
    outputs = base_model(**inputs)

print("last_hidden_state shape:", outputs.last_hidden_state.shape)
# (batch_size, sequence_length, hidden_size)

Input IDs: tensor([[ 464, 2003,  286, 9552,  318]])
last_hidden_state shape: torch.Size([1, 5, 768])


In [29]:
# Forward pass with the Causal LM head
with torch.no_grad():
    lm_outputs = lm_model(**inputs)

print("Logits shape:", lm_outputs.logits.shape)
# (batch_size, sequence_length, vocab_size)

# Next-token prediction for the last position
next_token_logits = lm_outputs.logits[0, -1, :]
next_token_id = next_token_logits.argmax()
print("Predicted next token ID:", next_token_id.item())
print("Predicted next token   :", tokenizer.decode(next_token_id))

Logits shape: torch.Size([1, 5, 50257])
Predicted next token ID: 8627
Predicted next token   :  uncertain


## 5. Device Placement

Models and tensors must live on the same device. Common patterns:

In [30]:
# 1. Move the whole model after loading
model = AutoModelForCausalLM.from_pretrained(model_name)
model = model.to(device)
print("Model device:", next(model.parameters()).device)

# 2. Load directly onto the device (saves a copy)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# 3. Using the device_map argument (useful for large models)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",          # requires accelerate
#     torch_dtype=torch.float16,  # optional half precision
# )

# Always move inputs to the same device
inputs = tokenizer("Hello world", return_tensors="pt").to(device)
print("Input device:", inputs["input_ids"].device)

with torch.no_grad():
    out = model(**inputs)
print("Logits device:", out.logits.device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Input device: cpu
Logits device: cpu


### Memory tips

- Use `torch_dtype=torch.float16` or `bfloat16` to halve memory.
- `device_map="auto"` (with `accelerate`) can spread large models across multiple GPUs or offload to CPU/disk.
- Call `model.eval()` and wrap inference in `torch.no_grad()` (or `torch.inference_mode()`) to disable gradients.

## 6. Inspecting Parameters & Memory

In [31]:
def model_summary(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Architecture     : {model.__class__.__name__}")
    print(f"Total parameters : {total:,}")
    print(f"Trainable params : {trainable:,}")
    # Rough memory estimate (float32)
    print(f"Approx size (fp32): {total * 4 / 1e6:.1f} MB")
    print(f"Approx size (fp16): {total * 2 / 1e6:.1f} MB")

model_summary(model)

Architecture     : GPT2LMHeadModel
Total parameters : 124,439,808
Trainable params : 124,439,808
Approx size (fp32): 497.8 MB
Approx size (fp16): 248.9 MB


In [32]:
# List the first few named parameters
print("Some named parameters:\n")
for i, (name, param) in enumerate(model.named_parameters()):
    if i >= 8:
        print("...")
        break
    print(f"{name:<50} {tuple(param.shape)}")

Some named parameters:

transformer.wte.weight                             (50257, 768)
transformer.wpe.weight                             (1024, 768)
transformer.h.0.ln_1.weight                        (768,)
transformer.h.0.ln_1.bias                          (768,)
transformer.h.0.attn.c_attn.weight                 (768, 2304)
transformer.h.0.attn.c_attn.bias                   (2304,)
transformer.h.0.attn.c_proj.weight                 (768, 768)
transformer.h.0.attn.c_proj.bias                   (768,)
...


## 7. Simple Generation with the Loaded Model

In [33]:
model.eval()
prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Prompt   :", prompt)
print("Generated:", tokenizer.decode(output_ids[0], skip_special_tokens=True))

Prompt   : The meaning of life is
Generated: The meaning of life is the essence of its existence", Aristotle said.

One of the most successful and powerful philosophy of all time was Aristotle's Ethics of Action, published


## 8. Loading Different Model Families (Examples)

The same `Auto*` API works across architectures. Uncomment the ones you want to try (they will download weights).

In [34]:
# --- BERT (encoder-only) ---
# tok = AutoTokenizer.from_pretrained("bert-base-uncased")
# model = AutoModel.from_pretrained("bert-base-uncased")

# --- DistilBERT (smaller & faster) ---
# tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# model = AutoModel.from_pretrained("distilbert-base-uncased")

# --- T5 (encoder-decoder) ---
# tok = AutoTokenizer.from_pretrained("t5-small")
# model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# --- A tiny modern causal LM ---
# tok = AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
# model = AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")

print("Uncomment any of the blocks above to load a different family.")
print("All of them use the same AutoTokenizer / AutoModel pattern.")

Uncomment any of the blocks above to load a different family.
All of them use the same AutoTokenizer / AutoModel pattern.


## 9. Best Practices Checklist

1. **Always read the model card** on the Hub before using a model.
2. Use `AutoTokenizer` and `AutoModel*` so your code stays architecture-agnostic.
3. Set `pad_token` when the original model lacks one (common for GPT-2).
4. Move both **model and inputs** to the same device.
5. Prefer `torch.inference_mode()` (or `torch.no_grad()`) for pure inference.
6. For large models, consider `torch_dtype=torch.float16` and `device_map="auto"`.
7. Keep the tokenizer and model from the **same checkpoint** to avoid token mismatches.

## 10. Summary

| Concept | What it does |
|---------|--------------|
| **Model Card** | Documentation + metadata on the Hub |
| **AutoConfig** | Load architecture settings without weights |
| **AutoTokenizer** | Load the matching tokenizer automatically |
| **AutoModel** | Load the base Transformer (hidden states) |
| **AutoModelForCausalLM** | Load a language-model head for generation |
| **Device placement** | `.to(device)` or `device_map="auto"` |

### Typical loading pattern

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

inputs = tokenizer("Hello", return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0]))
```

---

**Next notebook:** [`02_pipelines_and_generation.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/02_huggingface_basics/02_pipelines_and_generation.ipynb)  
Pipeline API · Text Generation · Sampling Strategies